# Imports + Config

In [7]:
import json, os, gc, time, random, copy, csv
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

DATA_BASE        = "/kaggle/input/YOUR_DATASET_NAME"   # ← adjust folder name
TRAIN_JSON       = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/train.json"
VAL_JSON         = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/validation.json"
TEST_JSON        = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/test.json"
SEMANTIC_MATCHES = f"/kaggle/input/datasets/d202511054/clef-2026-checkthat-task-2-data/my data/semantic_matches.json"
TRAIN_EMB_PATH   = f"/kaggle/input/datasets/d202511054/qwen3-embeddings/train_qwen3_embeddings.npz"
VAL_EMB_PATH     = f"/kaggle/input/datasets/d202511054/qwen3-embeddings/val_qwen3_embeddings.npz"
TEST_EMB_PATH    = f"/kaggle/input/datasets/d202511054/qwen3-embeddings/test_qwen3_embeddings.npz"

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLAIM_FIELD        = "claim"
TRACE_TEXT_FIELD   = "reasoning_traces"
VERDICT_LIST_FIELD = "Verdict_list"

label_map         = {"False": 0, "True": 1, "Conflicting": 2}
reverse_label_map = {0: "False", 1: "True", 2: "Conflicting"}

BATCH_SIZE          = 16
MAX_ITERS            = 100
LEARNING_RATE        = 1e-3
EARLY_STOP_PATIENCE  = 20
WEIGHT_DECAY         = 1e-5
GRAD_CLIP_NORM       = 1.0
TRACE_MAX            = 20
SCALAR_DIM           = 7
EXPANDED_SCALAR_DIM  = 1024
ATTN_HIDDEN          = 256
HIDDEN_DIMS          = [512, 256, 128]
OUTPUT_DIM           = 3
DEVICE               = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_SEED          = 42

def set_seed(seed=RANDOM_SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print(f"Device: {DEVICE}  GPUs: {torch.cuda.device_count()}")

Device: cuda  GPUs: 2


# Load JSON + ID Mappings


In [8]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def numpy_to_python(obj):
    if isinstance(obj, np.integer):    return int(obj)
    if isinstance(obj, np.floating):   return float(obj)
    if isinstance(obj, np.ndarray):    return obj.tolist()
    if isinstance(obj, dict):          return {k: numpy_to_python(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [numpy_to_python(i) for i in obj]
    return obj

_train_data       = load_json(TRAIN_JSON)
_val_data         = load_json(VAL_JSON)
_test_data        = load_json(TEST_JSON)
_semantic_matches = load_json(SEMANTIC_MATCHES)

def create_train_id_to_item():
    claim_to_item = {item[CLAIM_FIELD]: item for item in _train_data}
    out = {}
    for m in _semantic_matches:
        if m["new_claim"] in claim_to_item:
            out[m["new_id"]] = claim_to_item[m["new_claim"]]
    return out

train_id_to_item = create_train_id_to_item()
val_id_to_item   = {idx: item for idx, item in enumerate(_val_data)}
test_id_to_item  = {item["query_id"]: item for item in _test_data}

print(f"Train: {len(train_id_to_item)}  Val: {len(val_id_to_item)}  Test: {len(test_id_to_item)}")

Train: 6400  Val: 1600  Test: 2558


# Load Cached Embeddings (as plain dict — thread-safety fix preserved)


In [9]:
train_emb_data = dict(np.load(TRAIN_EMB_PATH, allow_pickle=False))
val_emb_data   = dict(np.load(VAL_EMB_PATH,   allow_pickle=False))
test_emb_data  = dict(np.load(TEST_EMB_PATH,  allow_pickle=False))

EMB_DIM = train_emb_data[list(train_emb_data.keys())[0]].shape[1]
COMBINED_DIM_BASE = EMB_DIM * 3 + EXPANDED_SCALAR_DIM
print(f"EMB_DIM = {EMB_DIM}  |  COMBINED_DIM (expanded) = {COMBINED_DIM_BASE}")

EMB_DIM = 4096  |  COMBINED_DIM (expanded) = 13312


# Feature Engineering (split_and_pad / verdict_features)


In [10]:
def split_and_pad(emb_array, max_traces=TRACE_MAX, emb_dim=None):
    if emb_dim is None: emb_dim = EMB_DIM
    claim_emb = emb_array[0].astype(np.float32)
    traces    = emb_array[1:].astype(np.float32)
    n = traces.shape[0]
    mask = np.zeros(max_traces, dtype=np.float32)
    if n < max_traces:
        pad = np.zeros((max_traces - n, emb_dim), dtype=np.float32)
        traces_padded = np.vstack([traces, pad])
        mask[:n] = 1.0
    else:
        traces_padded = traces[:max_traces]
        mask[:max_traces] = 1.0
    return claim_emb, traces_padded, mask

def verdict_features(verdict_list, max_traces=TRACE_MAX):
    counts = {0:0, 1:0, 2:0}
    for v in verdict_list:
        if v in label_map: counts[label_map[v]] += 1
    total = sum(counts.values())
    if total == 0: return np.zeros(SCALAR_DIM, dtype=np.float32)
    frac_false, frac_true, frac_conf = counts[0]/total, counts[1]/total, counts[2]/total
    majority_frac  = max(counts.values()) / total
    distinct_ratio = sum(1 for c in counts.values() if c > 0) / 3.0
    fracs   = [frac_false, frac_true, frac_conf]
    entropy = -sum(p*np.log(p+1e-9) for p in fracs if p > 0)
    num_traces_norm = min(total / max_traces, 1.0)
    return np.array([frac_false, frac_true, frac_conf, majority_frac,
                     distinct_ratio, entropy, num_traces_norm], dtype=np.float32)

# Datasets + DataLoaders


In [11]:
class BaseDataset(Dataset):
    label_key = "label"
    def __getitem__(self, idx):
        id_ = self.ids[idx]
        emb_array = self.emb_data[f"id_{id_}"]
        claim_emb, traces_padded, mask = split_and_pad(emb_array)
        item = self.id_to_item[id_]
        label = label_map[item[self.label_key]]
        scalars = verdict_features(item.get(VERDICT_LIST_FIELD, []))
        return (torch.from_numpy(claim_emb), torch.from_numpy(traces_padded),
                torch.from_numpy(mask), torch.from_numpy(scalars),
                torch.tensor(label, dtype=torch.long))
    def __len__(self): return len(self.ids)

class TrainDataset(BaseDataset):
    label_key = "label"
    def __init__(self):
        self.emb_data = train_emb_data
        self.id_to_item = train_id_to_item
        all_ids = sorted(int(k.split("_")[1]) for k in self.emb_data if k.startswith("id_"))
        self.ids = [i for i in all_ids if i in self.id_to_item]

class ValDataset(BaseDataset):
    label_key = "label"
    def __init__(self):
        self.emb_data = val_emb_data
        self.id_to_item = val_id_to_item
        self.ids = sorted(self.id_to_item.keys())

class TestDataset(BaseDataset):
    label_key = "Label"
    def __init__(self):
        self.emb_data = test_emb_data
        self.id_to_item = test_id_to_item
        self.ids = sorted(self.id_to_item.keys())

set_seed()
train_dataset = TrainDataset()
val_dataset   = ValDataset()
test_dataset  = TestDataset()
print(f"Train={len(train_dataset.ids)} Val={len(val_dataset.ids)} Test={len(test_dataset.ids)}")

from collections import Counter
_train_label_list = [label_map[train_dataset.id_to_item[i]["label"]] for i in train_dataset.ids]
_label_counts = Counter(_train_label_list)
_total = sum(_label_counts.values())
class_weights = torch.tensor([_total/(3.0*_label_counts[c]) for c in range(3)], dtype=torch.float32).to(DEVICE)
print(f"Class weights: {class_weights.tolist()}")

_dl_kw = dict(num_workers=0, pin_memory=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  **_dl_kw)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, **_dl_kw)

Train=6400 Val=1600 Test=2558
Class weights: [0.5739395618438721, 1.8156027793884277, 1.4146772623062134]


# Attention/Disagreement + Verdict Expanders


In [12]:
class AttentionPool(nn.Module):
    def __init__(self, dim, hidden=ATTN_HIDDEN):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, traces, mask):
        scores = self.attn(traces).squeeze(-1)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = torch.softmax(scores, dim=1)
        return torch.bmm(weights.unsqueeze(1), traces).squeeze(1)

def masked_std(traces, mask, eps=1e-6):
    mask_f = mask.unsqueeze(-1)
    count = mask_f.sum(dim=1).clamp(min=1.0)
    mean = (traces * mask_f).sum(dim=1) / count
    var = ((traces - mean.unsqueeze(1))**2 * mask_f).sum(dim=1) / count
    return torch.sqrt(var + eps)

class LearnableProjection(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, out_dim=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.ReLU(),
                                  nn.Linear(256, out_dim), nn.ReLU())
    def forward(self, x): return self.proj(x)

class VerdictAutoencoder(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, hidden=256, latent=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                     nn.Linear(hidden, latent), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(),
                                     nn.Linear(hidden, in_dim))
    def forward(self, x):
        z = self.encoder(x); return self.decoder(z), z
    def encode(self, x): return self.encoder(x)

class AEEncoderExpander(nn.Module):
    def __init__(self, ae): super().__init__(); self.encoder = ae.encoder
    def forward(self, x): return self.encoder(x)

class CyclicDuplication(nn.Module):
    def __init__(self, in_dim=SCALAR_DIM, out_dim=EXPANDED_SCALAR_DIM):
        super().__init__()
        self.out_dim = out_dim
        self.repeats = (out_dim + in_dim - 1) // in_dim
    def forward(self, x): return x.repeat(1, self.repeats)[:, :self.out_dim]

print("Modules defined.")

Modules defined.


# Retrain AE (lost on session close, fast to rebuild)


In [13]:
def collect_verdict_features_all_splits():
    vf_list = []
    for ds in [train_dataset, val_dataset, test_dataset]:
        for id_ in ds.ids:
            item = ds.id_to_item[id_]
            vf_list.append(verdict_features(item.get(VERDICT_LIST_FIELD, [])))
    return np.array(vf_list, dtype=np.float32)

_all_vf_np = collect_verdict_features_all_splits()
print(f"Collected {_all_vf_np.shape[0]} verdict vectors for AE pretraining")

def pretrain_ae(verdict_np, n_epochs=100, lr=1e-3, batch_size=64, device=DEVICE):
    set_seed()
    ae = VerdictAutoencoder().to(device)
    opt = optim.Adam(ae.parameters(), lr=lr, weight_decay=1e-5)
    mse = nn.MSELoss()
    X = torch.tensor(verdict_np, dtype=torch.float32)
    ldr = DataLoader(torch.utils.data.TensorDataset(X), batch_size=batch_size, shuffle=True)
    best_loss, best_state = float("inf"), None
    ae.train()
    for epoch in range(n_epochs):
        total = 0.0
        for (xb,) in ldr:
            xb = xb.to(device)
            opt.zero_grad()
            recon, _ = ae(xb)
            loss = mse(recon, xb)
            loss.backward(); opt.step()
            total += loss.item()
        avg = total / len(ldr)
        if avg < best_loss:
            best_loss = avg
            best_state = copy.deepcopy(ae.state_dict())
    ae.load_state_dict(best_state)
    ae.eval()
    print(f"AE pretrain done. Best MSE: {best_loss:.7f}")
    return ae

_pretrained_ae = pretrain_ae(_all_vf_np)
torch.save(_pretrained_ae.state_dict(), f"{OUTPUT_DIR}/pretrained_ae.pt")  # save this time!
print(f"AE weights saved -> {OUTPUT_DIR}/pretrained_ae.pt")

Collected 10558 verdict vectors for AE pretraining
AE pretrain done. Best MSE: 0.0000081
AE weights saved -> /kaggle/working/pretrained_ae.pt


# Configurable Model (No-Attention / No-Disagreement toggle)


In [14]:
class ConfigurableExpandedTraceMLP(nn.Module):
    def __init__(self, exp_config, verdict_expander, emb_dim=None,
                 expanded_scalar_dim=EXPANDED_SCALAR_DIM, hidden_dims=None, output_dim=OUTPUT_DIM):
        super().__init__()
        if emb_dim is None: emb_dim = EMB_DIM
        if hidden_dims is None: hidden_dims = HIDDEN_DIMS
        self.use_pool  = exp_config.get("use_pool", True)
        self.use_disag = exp_config.get("use_disagreement", True)
        self.verdict_expander = verdict_expander
        if self.use_pool:
            self.attn_pool = AttentionPool(emb_dim)
        combined_dim = emb_dim
        if self.use_pool:  combined_dim += emb_dim
        if self.use_disag: combined_dim += emb_dim
        combined_dim += expanded_scalar_dim
        self.combined_dim = combined_dim
        layers = [nn.BatchNorm1d(combined_dim)]
        prev = combined_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.3)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.mlp = nn.Sequential(*layers)

    def forward(self, claim, traces, mask, scalars):
        parts = [claim]
        if self.use_pool:  parts.append(self.attn_pool(traces, mask))
        if self.use_disag: parts.append(masked_std(traces, mask))
        parts.append(self.verdict_expander(scalars))
        return self.mlp(torch.cat(parts, dim=1))

print("ConfigurableExpandedTraceMLP ready.")

ConfigurableExpandedTraceMLP ready.


# Train/Eval/Report functions + Runner


In [15]:
def train_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train(); total_loss = 0.0
    for claim, traces, mask, scalars, labels in loader:
        claim, traces, mask, scalars, labels = (
            claim.to(device, non_blocking=True), traces.to(device, non_blocking=True),
            mask.to(device, non_blocking=True), scalars.to(device, non_blocking=True),
            labels.to(device, non_blocking=True))
        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                logits = model(claim, traces, mask, scalars)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer); scaler.update()
        else:
            logits = model(claim, traces, mask, scalars)
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval(); total_loss = 0.0
    all_preds, all_labels = [], []
    for claim, traces, mask, scalars, labels in loader:
        claim, traces, mask, scalars, labels = (
            claim.to(device, non_blocking=True), traces.to(device, non_blocking=True),
            mask.to(device, non_blocking=True), scalars.to(device, non_blocking=True),
            labels.to(device, non_blocking=True))
        logits = model(claim, traces, mask, scalars)
        total_loss += criterion(logits, labels).item()
        all_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
    acc = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    return total_loss / len(loader), acc, all_preds, all_labels

def generate_report(all_labels, all_preds, set_name="VAL", save_prefix=None):
    all_labels = np.array(all_labels, dtype=int); all_preds = np.array(all_preds, dtype=int)
    cm = confusion_matrix(all_labels, all_preds, labels=[0,1,2])
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
    recall    = recall_score(all_labels, all_preds, average=None, zero_division=0)
    f1        = f1_score(all_labels, all_preds, average=None, zero_division=0)
    macro_p, macro_r, macro_f1 = float(np.mean(precision)), float(np.mean(recall)), float(np.mean(f1))
    wp  = float(precision_score(all_labels, all_preds, average="weighted", zero_division=0))
    wr  = float(recall_score(all_labels, all_preds, average="weighted", zero_division=0))
    wf1 = float(f1_score(all_labels, all_preds, average="weighted", zero_division=0))
    print(f"\n{set_name}: Acc={acc:.4f} MacroF1={macro_f1:.4f} MacroRecall={macro_r:.4f} WeightedF1={wf1:.4f}")
    pfx = save_prefix if save_prefix else set_name.lower().replace(" ", "_")
    try:
        fig, ax = plt.subplots(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                    xticklabels=list(reverse_label_map.values()), yticklabels=list(reverse_label_map.values()))
        ax.set_title(f"{set_name} — Confusion Matrix"); ax.set_ylabel("True"); ax.set_xlabel("Pred")
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/{pfx}_confusion_matrix.png", dpi=100, bbox_inches="tight")
        plt.close()
    except Exception as e:
        print(f"plot failed: {e}")
    return {"accuracy": float(acc), "cm": cm.tolist(),
            "precision": numpy_to_python(precision), "recall": numpy_to_python(recall), "f1": numpy_to_python(f1),
            "avg_precision": macro_p, "avg_recall": macro_r, "avg_f1": macro_f1, "macro_f1": macro_f1,
            "weighted_precision": wp, "weighted_recall": wr, "weighted_f1": wf1}

def run_configurable_expanded_experiment(exp_name, exp_config, verdict_expander,
                                         train_loader, val_loader, test_loader,
                                         weights, device=DEVICE, use_amp=True):
    print(f"\n{'#'*72}\n# EXPERIMENT : {exp_name}\n{'#'*72}")
    set_seed()
    verdict_expander = verdict_expander.to(device)
    model = ConfigurableExpandedTraceMLP(exp_config, verdict_expander).to(device)
    _cdim = model.combined_dim
    print(f"combined_dim={_cdim}  params={sum(p.numel() for p in model.parameters()):,}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=weights)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    scaler = GradScaler() if (use_amp and device.startswith("cuda")) else None
    best_val_loss, patience_counter, best_state = float("inf"), 0, None
    train_losses, val_losses = [], []
    t0 = time.time()
    for epoch in range(MAX_ITERS):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion, device)
        train_losses.append(tr_loss); val_losses.append(va_loss)
        scheduler.step(va_loss)
        print(f"  Ep {epoch+1:3d}/{MAX_ITERS} | Train {tr_loss:.4f} | Val {va_loss:.4f} | ValAcc {va_acc:.4f}")
        if va_loss < best_val_loss:
            best_val_loss, patience_counter = va_loss, 0
            _base = model.module if isinstance(model, nn.DataParallel) else model
            best_state = copy.deepcopy(_base.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}"); break
    elapsed = time.time() - t0
    _base = model.module if isinstance(model, nn.DataParallel) else model
    _base.load_state_dict(best_state)
    pfx = exp_name.lower().replace(" ", "_")
    tr_f,_,tr_p,tr_l = evaluate(model, train_loader, criterion, device)
    va_f,_,va_p,va_l = evaluate(model, val_loader,   criterion, device)
    te_f,_,te_p,te_l = evaluate(model, test_loader,  criterion, device)
    tr_m = generate_report(tr_l, tr_p, f"{exp_name} TRAIN", f"{pfx}_train")
    va_m = generate_report(va_l, va_p, f"{exp_name} VAL",   f"{pfx}_val")
    te_m = generate_report(te_l, te_p, f"{exp_name} TEST",  f"{pfx}_test")
    del model; gc.collect(); torch.cuda.empty_cache()
    return {"experiment": exp_name,
            "config": {**exp_config, "expanded_scalar_dim": EXPANDED_SCALAR_DIM, "combined_dim": _cdim},
            "train_loss": float(tr_f), "val_loss": float(va_f), "test_loss": float(te_f),
            "train_metrics": tr_m, "val_metrics": va_m, "test_metrics": te_m,
            "train_losses": [float(x) for x in train_losses], "val_losses": [float(x) for x in val_losses],
            "training_time_s": float(elapsed)}

print("Runner ready.")

Runner ready.


# Run All 6 Experiments


In [16]:
CFG_C = dict(use_pool=False, use_disagreement=True)   # No Attention
CFG_D = dict(use_pool=True,  use_disagreement=False)  # No Disagreement

EXPANDERS = {
    "Dup":  CyclicDuplication,
    "AE":   lambda: AEEncoderExpander(_pretrained_ae),
    "Proj": LearnableProjection,
}

new_experiment_results = []

for _ename, _efactory in EXPANDERS.items():
    new_experiment_results.append(run_configurable_expanded_experiment(
        exp_name=f"C_NoAttn_{_ename}_1024", exp_config=CFG_C,
        verdict_expander=_efactory(),
        train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        weights=class_weights))

for _ename, _efactory in EXPANDERS.items():
    new_experiment_results.append(run_configurable_expanded_experiment(
        exp_name=f"D_NoDisag_{_ename}_1024", exp_config=CFG_D,
        verdict_expander=_efactory(),
        train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
        weights=class_weights))

print(f"\n✓ {len(new_experiment_results)} new experiments done.")


########################################################################
# EXPERIMENT : C_NoAttn_Dup_1024
########################################################################
combined_dim=9216  params=4,903,939
  Ep   1/100 | Train 0.9842 | Val 1.2848 | ValAcc 0.5706
  Ep   2/100 | Train 0.9212 | Val 2.1481 | ValAcc 0.5706
  Ep   3/100 | Train 0.8695 | Val 1.8289 | ValAcc 0.5706
  Ep   4/100 | Train 0.8225 | Val 1.4930 | ValAcc 0.5706
  Ep   5/100 | Train 0.8073 | Val 2.6979 | ValAcc 0.5706
  Ep   6/100 | Train 0.6955 | Val 2.8625 | ValAcc 0.5706
  Ep   7/100 | Train 0.6469 | Val 1.8353 | ValAcc 0.5744
  Ep   8/100 | Train 0.5903 | Val 4.2359 | ValAcc 0.5706
  Ep   9/100 | Train 0.5552 | Val 2.8610 | ValAcc 0.5713
  Ep  10/100 | Train 0.4661 | Val 2.2984 | ValAcc 0.5775
  Ep  11/100 | Train 0.4473 | Val 4.1515 | ValAcc 0.5706
  Ep  12/100 | Train 0.3910 | Val 5.3649 | ValAcc 0.5706
  Ep  13/100 | Train 0.3874 | Val 4.4791 | ValAcc 0.5706
  Ep  14/100 | Train 0.3644 | Val 3.9218 | 

# Final Sorted Summary

In [17]:
summary_rows = []
for r in new_experiment_results:
    test_f1     = float(r["test_metrics"]["macro_f1"])
    test_recall = float(r["test_metrics"]["avg_recall"])
    summary_rows.append({
        "Experiment": r["experiment"],
        "Test_Macro_F1": test_f1,
        "Test_Macro_Recall": test_recall,
        "Test_Average": (test_f1 + test_recall) / 2,
    })
summary_rows = sorted(summary_rows, key=lambda x: x["Test_Average"], reverse=True)

print("\n" + "="*90)
print(f"NEW EXPERIMENTS RESULTS — {len(summary_rows)} (Sorted by Test Average)")
print("="*90)
print(f"{'Rank':<5}{'Experiment':<28}{'Macro F1':>12}{'Macro Recall':>16}{'Average':>12}")
print("-"*90)
for idx, row in enumerate(summary_rows, start=1):
    print(f"{idx:<5}{row['Experiment']:<28}{row['Test_Macro_F1']:>12.4f}"
          f"{row['Test_Macro_Recall']:>16.4f}{row['Test_Average']:>12.4f}")
print("="*90)

csv_path = f"{OUTPUT_DIR}/no_attn_no_disag_1024_summary.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["Experiment","Test_Macro_F1","Test_Macro_Recall","Test_Average"])
    writer.writeheader(); writer.writerows(summary_rows)
with open(f"{OUTPUT_DIR}/no_attn_no_disag_1024_full.json", "w") as f:
    json.dump(numpy_to_python(new_experiment_results), f, indent=2)
print(f"\nSaved -> {csv_path}")


NEW EXPERIMENTS RESULTS — 6 (Sorted by Test Average)
Rank Experiment                      Macro F1    Macro Recall     Average
------------------------------------------------------------------------------------------
1    D_NoDisag_AE_1024                 0.5878          0.6143      0.6010
2    C_NoAttn_Dup_1024                 0.5971          0.5857      0.5914
3    D_NoDisag_Dup_1024                0.5810          0.5861      0.5835
4    D_NoDisag_Proj_1024               0.5223          0.5386      0.5305
5    C_NoAttn_Proj_1024                0.5257          0.5181      0.5219
6    C_NoAttn_AE_1024                  0.5270          0.5145      0.5207

Saved -> /kaggle/working/no_attn_no_disag_1024_summary.csv


In [18]:
print("end")

end
